In [1]:

import lib as fn # подгрузка используемыз библиотек
import parsing_SATELLITES_COORDS as pi # парсер координат спутников
import numpy as np

# геодезические координаты базовой станции
X_geodez = 3041464.0
Y_geodez = -815026.0
Z_geodez = 5527999.0

station = (X_geodez, Y_geodez, Z_geodez)

# поиск созвездия доступных спутников

# BRD400DLR_S_20260010000_01D_MN.rnx
filename ='BRD400DLR_S_20260010000_01D_MN.rnx.gz'

satellites = pi.parse_rinex(filename)

# отбор спутников, которые использует потребитель

user_satellites = ['R23','R15','R17','R26'] # список номеров спутников которые использует потребитель
result_satellite = [] # список словарей спутников для рассчета координат
satellites_coords = []

for satellite in satellites:
    if satellite['R'] in user_satellites:
        result_satellite.append(satellite)

        satellites_coords.append([
            float(np.asarray(satellite['x']).ravel()[0]),
            float(np.asarray(satellite['y']).ravel()[0]),
            float(np.asarray(satellite['z']).ravel()[0])
        ])

satellites_coords = np.array(satellites_coords, dtype=float)

# рассчет координат по спутникам 
res = fn.location(station, satellites_coords)

# вычисление вектора ошибки
delta_X = res[0] -  X_geodez
delta_Y = res[1] -  Y_geodez
delta_Z = res[2] -  Z_geodez

print(f' истинные координаты: X = {X_geodez} Y = {Y_geodez} Z = {Z_geodez } \n')
print(f' координаты по спутникам: X = {res[0]} Y = {res[1]} Z = {res[2] } \n')
print(f'dX = {delta_X } dY = {delta_Y} dZ = {delta_Z}')


 истинные координаты: X = 3041464.0 Y = -815026.0 Z = 5527999.0 

 координаты по спутникам: X = 3041464.0689468235 Y = -815041.8775292666 Z = 5528000.346324269 

dX = 0.06894682347774506 dY = -15.877529266639613 dZ = 1.3463242687284946


In [2]:
import gzip


def parser_ionex(filename, target_lat: float, target_lon: float) -> float:
    er_lat = 0.4
    exponent = -1  # По умолчанию для большинства IONEX

    with gzip.open(filename, 'rt') as f:
        for line in f:
            if 'LAT/LON1/LON2/DLON/H' in line:
                parts = line.strip().split()

                if abs(float(parts[0]) - target_lat) < er_lat:
                    lat = float(parts[0])
                    lon1 = float(parts[1])
                    lon2 = float(parts[2])
                    dlon = float(parts[3])

                    if lon1 <= target_lon <= lon2:
                        idx = int(round((target_lon - lon1) / dlon))

                        i = 0

                        # читаем следующие строки, там идут значения
                        for data_line in f:
                            if (
                                'LAT/LON1/LON2/DLON/H' in data_line
                                or 'END OF TEC MAP' in data_line
                                or 'START OF TEC MAP' in data_line
                            ):
                                break

                            for element in data_line.strip().split():
                                if i == idx:
                                    value = float(element) * (10 ** exponent)
                                    print(value)
                                    return value

                                i += 1

    return float('nan')


if __name__ == '__main__':
    filename = 'ROBR007A00.26I.gz'
    target_lat = 39.5
    target_lon = 1
    parser_ionex(filename, target_lat, target_lon) 

9.200000000000001


In [ ]:
import gzip



def parser_ionex(filename, target_lat: float, target_lon: float) -> float:

    exponent = -1 # По умолчанию для большинства IONEX

    with gzip.open(filename, 'rt') as f:

        for line in f:
            print(line)


if __name__ == '__main__':
    filename = 'ROBR007A00.26I.gz'
    target_lat = 1
    target_lon = 1
    parser_ionex(filename, target_lat, target_lon) 

In [7]:

import requests as rq
import coordinates_pb2 as c_pb2

URL = "http://127.0.0.1:5000/correction"

msg = c_pb2.Coordinates(x=1, y=2, z=3)

try:
    response = rq.post(
        URL,
        data=msg.SerializeToString(),
        headers={
            "Content-Type": "application/x-protobuf",
            "Accept": "application/x-protobuf",
        },
        timeout=10,
    )

    print("Статус-код:", response.status_code)

    response.raise_for_status()

    result = c_pb2.Coordinates()
    result.ParseFromString(response.content)

    print(f"x={result.x}, y={result.y}, z={result.z}")

except rq.RequestException as e:
    print("Ошибка HTTP-запроса:", e)
except Exception as e:
    print("Ошибка обработки ответа:", e)


Статус-код: 500
Ошибка HTTP-запроса: 500 Server Error: INTERNAL SERVER ERROR for url: http://127.0.0.1:5000/correction


In [18]:
import data_satellite_pb2 as ds_pb2

satellite= ds_pb2.Satellite()
satellite.x = 1
satellite.y = 1

binary_data = satellite.SerializeToString()


# десереализауия
satellite = ds_pb2.Satellite()
satellite.ParseFromString(binary_data)
print(satellite)


x: 1
y: 1



In [5]:

import requests as rq
import coordinates_pb2 as c_pb2

URL = "http://127.0.0.1:5001/correct"

msg = c_pb2.Coordinates(x=1, y=2, z=3)


response = rq.post(
        URL,
        data=msg.SerializeToString(),
        headers={
            "Content-Type": "application/x-protobuf",
            "Accept": "application/x-protobuf",
        },
        timeout=1,
    )

print("Статус-код:", response.status_code)

URL = "http://127.0.0.1:5000/"

response = rq.get(URL)

print(f'статус центрального сервера {response.status_code}')

# result = c_pb2.Coordinates()
# result.ParseFromString(response.content)

# print(f"x={result.x}, y={result.y}, z={result.z}")



Статус-код: 200
статус центрального сервера 200
